# MATH 4210 GAN experiments
Use **Runtime → Change runtime type → GPU**. This notebook runs the repository package rather than duplicating its training code. Start with the smoke matrix. Full experiments use 16 baseline runs and one learning-rate follow-up. Colab resources vary, so preserve checkpoints on Drive.


In [ ]:
from pathlib import Path
import io, zipfile, subprocess, sys, shutil
from google.colab import files, drive
PROJECT = Path('/content/math4210_gan')
PROJECT.mkdir(exist_ok=True)
print('Upload deliverables/gan_project_source.zip from your local project.')
uploaded = files.upload()
archive_name = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(io.BytesIO(uploaded[archive_name])) as archive:
    for name in archive.namelist():
        target = (PROJECT / name).resolve()
        if PROJECT.resolve() not in target.parents:
            raise ValueError('Archive path escapes the project directory')
    archive.extractall(PROJECT)
assert (PROJECT / 'pyproject.toml').exists(), 'Upload the source ZIP produced by scripts/package_source.py'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT)], check=True)


## Persistent storage and recovery
Authorize Drive access in the standard Colab dialog. Checkpoints and result files go in the folder below. Training data stays on the runtime disk for faster reads. Rerun the restore cell after reconnecting to a new runtime.


In [ ]:
drive.mount('/content/drive')
DURABLE = Path('/content/drive/MyDrive/math4210_gan/runs')
LOCAL_RUNS = PROJECT / 'runs'
DURABLE.mkdir(parents=True, exist_ok=True)
LOCAL_RUNS.mkdir(exist_ok=True)
# Restore only if the local run is absent. Never overwrite a newer local run.
for kind in ('smoke', 'baseline', 'followup'):
    folder = DURABLE / kind
    if folder.exists():
        for run in folder.iterdir():
            local = LOCAL_RUNS / kind / run.name
            if run.is_dir() and not local.exists():
                shutil.copytree(run, local)
import torch, torchvision
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
assert torch.cuda.is_available(), 'Choose a GPU runtime before starting training.'
print('GPU:', torch.cuda.get_device_name(0))


## Run experiments
The first execution defaults to smoke tests. Change MODE to 'baseline' for all 16 baseline runs or 'all' to include the follow-up. Set DATASETS to divide work among members. The follow-up requires all evaluated baselines in the same run tree. Completed training is skipped and interrupted training resumes automatically.


In [ ]:
MODE = 'smoke'  # 'baseline', 'followup', or 'all'
DATASETS = ['gaussian', 'gaussian_mixture', 'circle', 'sine', 'mnist', 'fashion_mnist', 'kmnist', 'cifar10']
command = [sys.executable, str(PROJECT / 'scripts/run_experiments.py'),
           '--mode', MODE, '--datasets', *DATASETS, '--device', 'cuda',
           '--output-root', str(LOCAL_RUNS), '--sync-dir', str(DURABLE)]
subprocess.run(command, cwd=PROJECT, check=True)


## Rebuild the comparison and inspect saved figures
This does not retrain models. Smoke and partial runs stay labeled. Inspect image grids before writing qualitative claims in each run's observations.md.


In [ ]:
subprocess.run([sys.executable, '-m', 'gan_project', 'summarize', '--runs', str(LOCAL_RUNS),
                '--output-dir', str(PROJECT / 'results')], cwd=PROJECT, check=True)
from IPython.display import display, Markdown, Image
display(Markdown((PROJECT / 'results/comparison.md').read_text()))
for figure in sorted(LOCAL_RUNS.glob('baseline/*/evaluation/comparison.png')):
    display(Image(filename=str(figure), width=800))


## Download the comparison
Per-run checkpoints and figures already copy to Drive at saved boundaries. This cell adds the aggregate report. Keep the original project ZIP and the source snapshot saved with each run. Use trusted checkpoints only.


In [ ]:
shutil.copytree(PROJECT / 'results', DURABLE.parent / 'results', dirs_exist_ok=True)
shutil.make_archive('/content/gan_results', 'zip', PROJECT / 'results')
files.download('/content/gan_results.zip')
